# TraceletCodeAgent (direct_prompt strategy, n_samples=1) on GAIA -- Qwen3.7-Plus

Same as `traceletReAct_directprompt_qwen37plus.ipynb`, but with `n_samples=1` instead of 3. See `traceletReAct_directprompt_n1.ipynb` for the ablation rationale. Uses distinct `output_file`/`pickle_dir` names so it doesn't collide with the n_samples=3 run.

In [1]:
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")

from dotenv import load_dotenv
load_dotenv()

from smolagents import OpenAIModel

model_name = "Qwen/Qwen3.7-Plus"
enable_thinking = False  # hardcoded (was interactive input()) so this notebook can run headlessly via nbconvert
# Together requires enable_thinking nested inside chat_template_kwargs for vLLM-served open-weight
# models (see naiveReAct.ipynb). client_kwargs timeout bounds a single API call so a stalled/hanging
# stream fails within 5 minutes instead of hanging indefinitely.
model = OpenAIModel(
    model_id=model_name,
    api_base="https://api.together.ai/v1/",
    api_key=os.environ["TOGETHER_API_KEY"],
    extra_body={"chat_template_kwargs": {"enable_thinking": enable_thinking}},
    client_kwargs={"timeout": 300.0},
)

In [2]:
# Reuse the standard GAIA tool stack, same as naiveReAct.ipynb
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

/Users/poorvag/Work/smolagents/.venv/lib/python3.14/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [3]:
from smolagents.monitoring import LogLevel
from smolagents.tracelet_agent import TraceletCodeAgent

agent = TraceletCodeAgent(
    tools=tools,
    model=model,
    max_steps=50,
    verbosity_level=LogLevel.INFO,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    n_samples=1,
    skeleton_strategy="direct_prompt",  # model emits the sentinel-marked skeleton itself
    stream_outputs=True,  # Qwen models via TogetherAI reject non-streaming requests outright
)

In [4]:
# Load GAIA validation set from HuggingFace
import pandas as pd
from common_setup import load_gaia_dataset

SET_TO_RUN = "validation"
eval_ds = load_gaia_dataset(set_to_run=SET_TO_RUN)

print(f"Loaded {len(eval_ds)} examples")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 165 examples


In [5]:
from common_setup import evaluate_agent, question_scorer

results = evaluate_agent(
    agent,
    eval_ds,
    ti_tool,
    visualizer,
    n_samples=50,  # quick validation of the model_output fix before a full 165-question rerun
    output_file=f"tracelet_direct_n1_react_{model_name}.jsonl",
    pickle_dir=f"tracelet_direct_n1_react_{model_name}",
)


[1/50] (cached) A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure w...
  ✗ | cached

[2/50] (cached) I’m researching species that became invasive after people who kept them as pets released them. There...
  ✗ | cached

[3/50] (cached) If we assume all articles published by Nature in 2020 (articles, only, not book reviews/columns, etc...
  ✗ | cached

[4/50] (cached) In Unlambda, what exact charcter or text needs to be added to correct the following code to output "...
  ✗ | cached

[5/50] (cached) If Eliud Kipchoge could maintain his record-making marathon pace indefinitely, how many thousand hou...
  ✗ | cached

[6/50] (cached) The attached spreadsheet shows the inventory for a movie and video game rental store in Seattle, Was...
  ✓ | cached

[7/50] (cached) How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? You can use...
  ✗ | cached

[8/50] (cached) The object in the British Museum's collection

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ It is 1999. Before you party like it is 1999, please assist me in settling a bet.                               │
│                                                                                                                 │
│ Fiona Apple and Paula Cole released albums prior to 1999. Of these albums, which didn't receive a letter grade  │
│ from Robert Christgau? Provide your answer as a comma delimited list of album titles, sorted alphabetically.    │
│                                                                                                                 │
╰─ OpenAIModel - Qwen/Qwen3.7-Plus ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Tracelet] 1 sentinel(s) found -- sampling 1 candidates.

[Tracelet] judge scores: [0.0] -- picked candidate 0.

...

[Tracelet] no tool-call arguments to sample this step -- direct execution (no fill-in sampling, no judge).

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("A Nightmare on Elm Street")                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: A Nightmare on Elm Street

[Step 14: Duration 4.49 seconds| Input tokens: 210,891 | Output tokens: 1,864]

  ✓ | 92.3s | 14 steps | tokens: TokenUsage(input_tokens=210891, output_tokens=1864, total_tokens=212755)


In [6]:
df = pd.DataFrame(results)
total = len(df)
correct = df["is_correct"].sum()

print("=== TraceletCodeAgent (direct_prompt, n_samples=1) GAIA Evaluation Results ===")
print(f"Overall accuracy:   {correct}/{total} = {correct/total:.1%}")
print(f"Avg time per question: {df['time_taken_seconds'].mean():.1f}s")
print(f"Avg steps per question: {df['num_steps'].mean():.1f}")

total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0)).mean()
print(f"Avg total tokens per question: {total_tokens:,.0f}")

print(f"\nTool usage (total calls across all questions):")
tool_usage_df = pd.DataFrame(df["tool_usage"].tolist()).sum().sort_values(ascending=False)
for tool, count in tool_usage_df.items():
    if count > 0:
        print(f"  {tool}: {int(count)}")

=== TraceletCodeAgent (direct_prompt, n_samples=1) GAIA Evaluation Results ===
Overall accuracy:   13/50 = 26.0%
Avg time per question: 1484.5s
Avg steps per question: 34.4
Avg total tokens per question: 1,088,872

Tool usage (total calls across all questions):
  web_search: 474
  visit_page: 175
  page_down: 64
  inspect_file_as_text: 47
  find_on_page_ctrl_f: 20
  final_answer: 19
